# Quantum Finance with Quantumium

This notebook demonstrates quantum computing applications in finance:

1. **Option Pricing** — European options via quantum amplitude estimation
2. **Portfolio Optimization** — Asset selection via QAOA
3. **Risk Analysis** — Value at Risk (VaR) estimation

In [ ]:
import numpy as np
from quantumium.finance import (
    european_option_price,
    black_scholes_price,
    quantum_portfolio_optimization,
    quantum_var,
)

## 1. European Option Pricing

Pricing financial derivatives is computationally intensive — classical Monte Carlo
methods converge as O(1/√N). Quantum amplitude estimation can achieve O(1/N),
a quadratic speedup.

We price a European call option and compare against the Black-Scholes analytical formula.

**Parameters:**
- Spot price S = $100
- Strike price K = $100 (at the money)
- Risk-free rate r = 5%
- Volatility σ = 20%
- Time to maturity T = 1 year

In [ ]:
S, K, r, vol, T = 100.0, 100.0, 0.05, 0.2, 1.0

# Black-Scholes reference price
bs_price = black_scholes_price(S, K, r, vol, T, "call")
print(f"Black-Scholes call price: ${bs_price:.4f}")
print()

# Quantum pricing with increasing qubit counts
print(f"{'Qubits':>6} {'Bins':>6} {'Quantum Price':>14} {'BS Price':>10} {'Rel Error':>10}")
print("-" * 50)

for n_qubits in [3, 4, 5, 6]:
    result = european_option_price(S, K, r, vol, T, "call", num_qubits=n_qubits)
    print(f"{n_qubits:>6} {result.num_price_bins:>6} "
          f"${result.quantum_price:>12.4f} "
          f"${result.classical_price:>8.4f} "
          f"{result.relative_error:>9.2%}")

print("\nMore qubits → finer price discretization → better accuracy.")

In [ ]:
# Put option pricing
print("Put option pricing (K=$100, at the money):")
print()
for n_qubits in [4, 5, 6]:
    result = european_option_price(S, K, r, vol, T, "put", num_qubits=n_qubits)
    bs_put = black_scholes_price(S, K, r, vol, T, "put")
    print(f"  {n_qubits} qubits: Quantum=${result.quantum_price:.4f}  BS=${bs_put:.4f}  "
          f"Error={result.relative_error:.2%}")

# Verify put-call parity: C - P = S - K*exp(-rT)
call_5 = european_option_price(S, K, r, vol, T, "call", num_qubits=5)
put_5 = european_option_price(S, K, r, vol, T, "put", num_qubits=5)
parity_lhs = call_5.quantum_price - put_5.quantum_price
parity_rhs = S - K * np.exp(-r * T)
print(f"\nPut-Call Parity Check (5 qubits):")
print(f"  C - P = ${parity_lhs:.4f}")
print(f"  S - K*exp(-rT) = ${parity_rhs:.4f}")

In [ ]:
# Sensitivity analysis: vary volatility
print("Call price sensitivity to volatility (5 qubits):")
print(f"{'σ':>6} {'Quantum':>10} {'BS':>10} {'Error':>8}")
print("-" * 38)
for sigma in [0.10, 0.15, 0.20, 0.25, 0.30, 0.40]:
    res = european_option_price(S, K, r, sigma, T, "call", num_qubits=5)
    bs = black_scholes_price(S, K, r, sigma, T, "call")
    print(f"{sigma:>6.0%} ${res.quantum_price:>8.4f} ${bs:>8.4f} {res.relative_error:>7.2%}")

## 2. Portfolio Optimization

Given a universe of assets with expected returns and a covariance matrix,
select a subset that maximizes return while minimizing risk.

This is formulated as a Quadratic Unconstrained Binary Optimization (QUBO)
problem and solved using QAOA.

$$\min_x \quad q \cdot x^T \Sigma x - \mu^T x + \lambda (\mathbf{1}^T x - B)^2$$

where x is a binary vector (1 = include asset, 0 = exclude).

In [ ]:
# Define a small asset universe
asset_names = ["Tech", "Healthcare", "Energy", "Finance"]
expected_returns = [0.12, 0.08, 0.10, 0.06]  # Annual expected returns

# Covariance matrix
covariance = [
    [0.04, 0.01, 0.005, 0.008],
    [0.01, 0.03, 0.004, 0.006],
    [0.005, 0.004, 0.035, 0.010],
    [0.008, 0.006, 0.010, 0.025],
]

print("Asset Universe:")
print(f"{'Asset':>12} {'Exp Return':>12} {'Volatility':>12}")
print("-" * 38)
for name, ret, var in zip(asset_names, expected_returns, np.diag(covariance)):
    print(f"{name:>12} {ret:>11.1%} {np.sqrt(var):>11.1%}")

In [ ]:
# Select 2 assets from 4 using quantum optimization
result = quantum_portfolio_optimization(
    expected_returns=expected_returns,
    covariance=covariance,
    budget=2,          # Select exactly 2 assets
    risk_factor=0.5,   # Balance between risk and return
    qaoa_layers=2,
    num_restarts=50,
)

print(f"Quantum Portfolio Optimization Result:")
print(f"  Selected assets: {[asset_names[i] for i in result.selection]}")
print(f"  Allocation:      {result.optimal_bitstring}")
print(f"  Expected return: {result.expected_return:.2%}")
print(f"  Portfolio risk:  {np.sqrt(result.risk):.2%} (volatility)")
print(f"  Objective value: {result.objective_value:.6f}")

In [ ]:
# Compare all possible 2-asset portfolios (brute force for validation)
from itertools import combinations

mu = np.array(expected_returns)
sigma = np.array(covariance)

print("All possible 2-asset portfolios:")
print(f"{'Portfolio':>25} {'Return':>8} {'Risk':>8} {'Objective':>10}")
print("-" * 55)

for combo in combinations(range(4), 2):
    x = np.zeros(4)
    x[list(combo)] = 1
    ret = mu @ x
    risk = x @ sigma @ x
    obj = 0.5 * risk - ret
    names = " + ".join(asset_names[i] for i in combo)
    marker = " ← QAOA" if set(combo) == set(result.selection) else ""
    print(f"{names:>25} {ret:>7.2%} {np.sqrt(risk):>7.2%} {obj:>10.6f}{marker}")

## 3. Risk Analysis — Value at Risk (VaR)

**Value at Risk (VaR)** answers: "What is the maximum loss at a given confidence level?"

For example, a 95% VaR of $10,000 means there's only a 5% chance of losing more than $10,000.

Quantum amplitude estimation can speed up VaR computation for complex portfolios.

In [ ]:
# Portfolio parameters
portfolio_mean = 0.05      # 5% expected return
portfolio_std = 0.15       # 15% volatility
confidence = 0.95          # 95% confidence level

# Quantum VaR (uses quantum amplitude estimation internally, classical VaR included in result)
q_result = quantum_var(
    mu=portfolio_mean,
    sigma=portfolio_std,
    confidence=confidence,
    num_qubits=5,
)

print(f"Portfolio: μ={portfolio_mean:.1%}, σ={portfolio_std:.1%}")
print(f"Confidence level: {confidence:.0%}")
print()
print(f"Classical VaR:  {q_result.var_classical:.4f}")
print(f"Classical CVaR: {q_result.cvar_classical:.4f}")
print(f"Quantum VaR:    {q_result.var_quantum:.4f}")
print(f"Quantum CVaR:   {q_result.cvar_quantum:.4f}")
print(f"Bins:           {q_result.num_bins}")

In [ ]:
# Quantum VaR with increasing qubit counts
print(f"{'Qubits':>6} {'Bins':>6} {'Q-VaR':>10} {'C-VaR':>10} {'Q-CVaR':>10} {'C-CVaR':>10}")
print("-" * 56)

for n_qubits in [3, 4, 5, 6]:
    res = quantum_var(
        mu=portfolio_mean,
        sigma=portfolio_std,
        confidence=confidence,
        num_qubits=n_qubits,
    )
    print(f"{n_qubits:>6} {res.num_bins:>6} "
          f"{res.var_quantum:>9.4f} {res.var_classical:>9.4f} "
          f"{res.cvar_quantum:>9.4f} {res.cvar_classical:>9.4f}")

print("\nMore qubits improve the discretization accuracy.")

In [ ]:
# Risk analysis at different confidence levels
print("VaR at different confidence levels (5 qubits):")
print(f"{'Confidence':>12} {'VaR':>10} {'CVaR':>10} {'Interpretation':>40}")
print("-" * 76)

for conf in [0.90, 0.95, 0.99]:
    res = quantum_var(mu=portfolio_mean, sigma=portfolio_std,
                      confidence=conf, num_qubits=5)
    interp = f"{(1-conf)*100:.0f}% chance of losing more than {res.var_quantum:.2%}"
    print(f"{conf:>11.0%} {res.var_quantum:>9.4f} {res.cvar_quantum:>9.4f} {interp:>40}")

## Summary

| Application | Classical | Quantum | Speedup |
|-------------|-----------|---------|--------|
| **Option Pricing** | Monte Carlo O(1/√N) | Amplitude Estimation O(1/N) | Quadratic |
| **Portfolio Optimization** | Brute force O(2^n) | QAOA approximate | Potential exponential |
| **Risk Analysis (VaR)** | Monte Carlo O(1/√N) | Amplitude Estimation O(1/N) | Quadratic |

These quantum finance algorithms are among the most promising near-term applications
of quantum computing, particularly for large portfolios where classical methods become
intractable.

For more details, see the [Quantumium finance module documentation](../docs/getting-started.md).